# Пространственный анализ предпосылок формирования локальной идентичности пойменных территорий Оренбурга

Практическое занятие № 2. НИР «Разработка метода выявления и учёта локальной идентичности пойменных территорий в градостроительном планировании».

Единица анализа — квартал города Оренбурга. Урал и Сакмара рассматриваются как части водно-зелёного каркаса. Карты показывают пространственные предпосылки локальной идентичности. Их интерпретация требует социокультурной проверки.

Запускайте разделы по очереди в ядре .venv. На первом запуске проверьте границу. Слои и графы формируются из локального PBF. Инструкция и нормативные источники приведены в [плане](../ORENBURG_RESEARCH.md).

In [ ]:
from datetime import datetime, timezone
from hashlib import sha256
from importlib.metadata import version
from pathlib import Path
import inspect
import json
import sys

import folium
import geopandas as gpd
import networkx as nx
import numpy as np
import pandas as pd
import shapely as shp
from IPython.display import display
from pyproj import CRS
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components, dijkstra
from scipy.spatial import cKDTree

from blocksnet.analysis.indicators import calculate_density_indicators
from blocksnet.analysis.morphotypes import get_spacematrix_morphotypes, get_strelka_morphotypes
from blocksnet.analysis.network.accessibility import area_accessibility
from blocksnet.analysis.provision import competitive_provision
from blocksnet.blocks.aggregation import aggregate_objects
from blocksnet.blocks.assignment import assign_land_use
from blocksnet.blocks.cutting import cut_urban_blocks, preprocess_urban_objects
from blocksnet.enums import LandUse
from blocksnet.preprocessing.imputing import impute_buildings
from blocksnet.relations import validate_accessibility_graph

def require(condition, message):
    if not condition:
        raise ValueError(message)

require(sys.version_info[:3] == (3, 12, 10), 'Выберите ядро Python 3.12.10 из .venv')
require(version('blocksnet') == '1.0.0a9', 'Требуется BlocksNet 1.0.0a9')
require({'self_supply', 'max_depth'} <= set(inspect.signature(competitive_provision).parameters),
        'Несовместимый API competitive_provision')

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from my_package.osm_snapshot import DRIVE_SPEEDS, ensure_graph, read_snapshot
from my_package.research_quality import calibrate_population, infer_entrances

WORK = ROOT / 'data' / 'orenburg'
INPUT, RAW, CACHE, OUT = (WORK / name for name in ('input', 'raw', 'cache', 'results'))
for folder in (INPUT, RAW, CACHE, OUT):
    folder.mkdir(parents=True, exist_ok=True)

# Готовый транспортный граф можно передать как локальную замену автомобильного.
CFG = dict(
    place='Оренбург, Оренбургская область, Россия',
    boundary_osm_id=None, boundary_confirmed=True,
    transport_mode=None, transport_source=None,
    transport_weight='time_min', transport_unit='minutes',
    floodplain_source=None, riparian_buffer_m=500, riparian_share_min=0.10,
    recreation_m2_person=11, walk_limit_m=1000, transport_limit_min=10,
    population_target=536_500, potential_entry_spacing_m=15,
    living_m2_person=30, default_floors=1, living_share=0.9,
    entrance_tolerance_m=15, snap_limit_m=100, network_margin_m=1500,
    max_matrix_mib=256, max_solver_pairs=2_000_000, max_route_labels=2_000_000,
    clusters=8, seed=42,
)
require(CFG['recreation_m2_person'] > 0 and CFG['walk_limit_m'] > 0
        and CFG['transport_limit_min'] > 0, 'Проверьте пороги')
VERSIONS = {name: version(name) for name in
            ('blocksnet', 'osmnx', 'geopandas', 'shapely', 'numpy', 'pandas', 'scipy', 'networkx', 'pyarrow', 'pulp', 'folium', 'pyproj')}
CACHE_REVISION = 1

def file_hash(path):
    with path.open('rb') as stream:
        from hashlib import file_digest
        return file_digest(stream, 'sha256').hexdigest()

def key(*values):
    payload = json.dumps((CACHE_REVISION, VERSIONS, values), sort_keys=True, default=str)
    return sha256(payload.encode()).hexdigest()[:24]

def stage(name, signature, build):
    # Манифест записывается последним. Повреждённый или неполный этап пересчитывается.
    folder = CACHE / f'{name}-{signature}'
    manifest = folder / 'manifest.json'
    if manifest.exists():
        try:
            spec = json.loads(manifest.read_text(encoding='utf-8'))
            if spec and all(file_hash(folder / f'{item}.parquet') == meta['sha256']
                            for item, meta in spec.items()):
                print(f'{name}: чтение кэша')
                return {item: (gpd if meta['spatial'] else pd).read_parquet(folder / f'{item}.parquet')
                        for item, meta in spec.items()}
        except (OSError, ValueError, KeyError):
            pass
    result = build()
    folder.mkdir(parents=True, exist_ok=True)
    spec = {}
    for item, frame in result.items():
        path = folder / f'{item}.parquet'
        temporary = path.with_suffix('.tmp')
        frame.to_parquet(temporary)
        temporary.replace(path)
        spec[item] = dict(spatial=isinstance(frame, gpd.GeoDataFrame), sha256=file_hash(path))
    temporary = manifest.with_suffix('.tmp')
    temporary.write_text(json.dumps(spec), encoding='utf-8')
    temporary.replace(manifest)
    return result

def shapes(frame, geometry_types, clip=True):
    require(frame.crs is not None, 'У слоя отсутствует CRS')
    result = frame.to_crs(METRIC).copy()
    result.geometry = result.geometry.make_valid()
    result = result.explode(index_parts=False, ignore_index=True)
    result = result[result.geom_type.isin(geometry_types) & ~result.geometry.is_empty]
    if clip:
        result = gpd.clip(result, city).explode(index_parts=False, ignore_index=True)
        result = result[result.geom_type.isin(geometry_types) & ~result.geometry.is_empty]
    return result.reset_index(drop=True)

POLY = ['Polygon', 'MultiPolygon']
LINE = ['LineString', 'MultiLineString']

def map_layers(layers):
    center = city.to_crs(4326).representative_point().iloc[0]
    result = folium.Map(location=[center.y, center.x], zoom_start=11, tiles='CartoDB positron')
    for label, frame, color in layers:
        if not frame.empty:
            folium.GeoJson(frame.to_crs(4326).to_json(), name=label,
                           style_function=lambda _, color=color: {'color': color, 'weight': 2, 'fillOpacity': 0.23}
                           ).add_to(result)
    folium.LayerControl().add_to(result)
    return result

quality_key = key('research-quality', file_hash(ROOT / 'my_package' / 'research_quality.py'),
                  CFG['population_target'], CFG['potential_entry_spacing_m'])

def show_map(result, name):
    folder = OUT / 'maps' / boundary_key / quality_key
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / f'{name}.html'
    result.save(str(path))
    print(f'HTML: {path}')
    display(result)

def thematic(frame, field, label):
    result = map_layers([('Граница', city, '#555555')])
    source = frame[[field, 'geometry']].to_crs(4326).copy()
    values = source[field].dropna()
    bins = np.unique(values.quantile([0.2, 0.4, 0.6, 0.8]).to_numpy()) if pd.api.types.is_numeric_dtype(values) and not values.empty else []
    categories = {v: i for i, v in enumerate(sorted(values.unique()))} if not pd.api.types.is_numeric_dtype(values) else {}
    palette = ['#f1eef6', '#bdc9e1', '#74a9cf', '#2b8cbe', '#045a8d']
    def style(feature):
        value = feature['properties'][field]
        idx = categories[value] % len(palette) if value in categories else int(np.searchsorted(bins, value)) if value is not None else 0
        return {'color': '#555', 'weight': 0.4, 'fillOpacity': 0.7,
                'fillColor': palette[idx] if value is not None else '#bdbdbd'}
    folium.GeoJson(source.to_json(), style_function=style,
                   tooltip=folium.GeoJsonTooltip(fields=[field], aliases=[label])).add_to(result)
    return result

## 1. Граница и открытые данные

При наличии положите официальную границу населённого пункта в input/city.geojson. Иначе используется сохранённый контур Nominatim. Сверьте карту с границей города: выдача может относиться к городскому округу. Если требуется, укажите идентификатор отношения OSM в формате R123. Входные слои и графы читаются из локального PBF. Состав и полнота тегов отражают фактическое картирование. © OpenStreetMap contributors.

In [ ]:
city_file = INPUT / 'city.geojson'
boundary_key = key('boundary', file_hash(city_file) if city_file.exists() else
                   (CFG['place'], CFG['boundary_osm_id']))

def build_boundary():
    if city_file.exists():
        boundary = gpd.read_file(city_file)
    else:
        raise ValueError('Нет сохранённой границы. Положите полигон города в input/city.geojson.')
    require(boundary.crs is not None and not boundary.empty, 'Не найдена граница города с CRS')
    require(boundary.geom_type.isin(POLY).all(), 'Ожидается полигональная граница города')
    geom = boundary.to_crs(4326).geometry.make_valid().union_all()
    require(geom.geom_type in POLY and geom.is_valid and not geom.is_empty, 'Некорректная граница')
    return {'city': gpd.GeoDataFrame(geometry=[geom], crs=4326)}

city = stage('boundary', boundary_key, build_boundary)['city']
METRIC = city.estimate_utm_crs()
require(METRIC is not None, 'Не удалось подобрать метрическую CRS')
city = city.to_crs(METRIC)
display(city.area.div(1e6).rename('Площадь границы, км²'))
show_map(map_layers([('Граница исследования', city, '#303030')]), 'city_boundary')

In [ ]:
import webbrowser

map_path = WORK / 'city_boundary.html'
map_layers([('Граница исследования', city, '#303030')]).save(str(map_path))
webbrowser.open(map_path.resolve().as_uri())

In [ ]:
require(CFG['boundary_confirmed'], 'Проверьте карту, установите boundary_confirmed=True и снова выполните конфигурацию')
TAGS = {
    'structures': {'building': True},
    'barriers': {'highway': ['motorway', 'trunk', 'primary', 'secondary', 'tertiary', 'residential', 'unclassified'],
                 'railway': ['rail', 'light_rail'], 'waterway': ['river', 'canal'],
                 'natural': ['water']},
    'landscape': {'landuse': True, 'leisure': ['park', 'garden', 'nature_reserve', 'recreation_ground'],
                  'natural': ['wood', 'grassland', 'scrub', 'wetland', 'beach', 'peak', 'cliff', 'rock', 'spring', 'tree'],
                  'place': ['square'], 'man_made': ['embankment'], 'footway': ['promenade']},
    'markers': {'historic': True, 'heritage': True,
                'tourism': ['museum', 'viewpoint', 'artwork', 'attraction']},
    'access_points': {'entrance': True, 'barrier': ['gate', 'entrance']},
    'crossings': {'highway': ['crossing', 'footway', 'pedestrian', 'steps'],
                  'railway': ['crossing', 'level_crossing']},
}
snapshot_file = RAW / 'orenburg.osm.pbf'
require(snapshot_file.is_file(), f'Положите распакованный PBF из BBBike в {snapshot_file}')
snapshot_sha256 = file_hash(snapshot_file)
reader_key = key(file_hash(ROOT / 'my_package' / 'osm_snapshot.py'), version('pyogrio'))
source_key = key('local-pbf', boundary_key, snapshot_sha256, reader_key, TAGS,
                 CFG['network_margin_m'], CFG['riparian_buffer_m'])
source_area = city.to_crs(4326).geometry.iloc[0]
network_extent = city.buffer(CFG['network_margin_m']).to_crs(4326).iloc[0]
river_extent = city.buffer(CFG['riparian_buffer_m']).to_crs(4326).iloc[0]

def fetch_osm():
    result = read_snapshot(snapshot_file, source_area, network_extent, river_extent, TAGS)
    decoded_utc = datetime.now(timezone.utc).isoformat()
    for frame in result.values():
        frame['retrieved_utc'] = decoded_utc
    return result

snapshot = stage('osm-pbf', source_key, fetch_osm)
osm = {name: snapshot[name] for name in TAGS}
display(pd.Series({name: len(frame) for name, frame in osm.items()}, name='Объекты OSM'))
print('Дорожных линий для графов:', len(snapshot['network_ways']))

def tag(frame, column):
    return frame.get(column, pd.Series('', index=frame.index, dtype='string')).fillna('')

def polygon_frame(geometries):
    return gpd.GeoDataFrame(geometry=list(geometries), crs=METRIC)

def safe_ratio(numerator, denominator):
    return numerator.div(denominator.where(denominator > 0))

## 2. Кварталы, использование территории и застройка

Дороги, железные дороги и вода из OSM задают геометрию кварталов. Площади считаются в местной UTM. Функциональные доли получаются пересечением с полигонами OSM после удаления перекрытий: при конфликте приоритет у меньшего контура, обычно более конкретного. Непокрытая площадь остаётся неизвестной. ID — хеш нормализованной геометрии в рабочей CRS. Изменение границы квартала меняет ID.

Этажность берётся из building:levels. Пропуски оцениваются с заданной этажностью. Жилые здания определяются по явным тегам и, для общих тегов building=yes, по жилой зоне. Население оценивается через жилую площадь. Плотность населения — человек на гектар всей площади квартала. Для контроля сохранены числа зданий с оценочной этажностью и без определённой функции.

| Поле | Метод и единица |
|---|---|
| site_area, footprint_area | Площадь квартала и сумма пятен зданий, м² |
| build_floor_area, living_area, non_living_area | Поэтажная, жилая и нежилая площади, м², с учётом импутации |
| fsi, gsi | Поэтажная площадь и площадь пятен, делённые на площадь квартала, м²/м² |
| mxi | Жилая доля поэтажной площади, 0–1 |
| l | Поэтажная площадь / площадь пятен, средняя этажность |
| osr | Площадь вне пятен / поэтажная площадь, м²/м² |
| share_living, share_non_living | Жилая и нежилая площади на м² пятен |
| spacematrix, strelka | Категории морфологии по l, fsi и mxi. Кластер Spacematrix зависит от выборки и параметров |

BlocksNet относит здание к кварталу по внутренней точке. Здания, пересекающие границы, и пробелы OSM создают погрешность. Для кварталов без застройки отношения с нулевым знаменателем отсутствуют.

Оценка населения из зданий сохраняется в population_original. Поле population откалибровано пропорционально к заданному общегородскому итогу 536 500 человек. Для точного целочисленного баланса остаток распределяется по наибольшим дробным долям. Калибровка сохраняет исходное пространственное распределение с точностью до округления. Исходная плотность сохраняется в population_ha_original.

In [ ]:
RULES = {
    'residential': LandUse.RESIDENTIAL, 'commercial': LandUse.BUSINESS, 'retail': LandUse.BUSINESS,
    'industrial': LandUse.INDUSTRIAL, 'garages': LandUse.INDUSTRIAL, 'brownfield': LandUse.INDUSTRIAL,
    'railway': LandUse.TRANSPORT, 'landfill': LandUse.SPECIAL, 'military': LandUse.SPECIAL,
    'cemetery': LandUse.SPECIAL, 'construction': LandUse.SPECIAL, 'religious': LandUse.SPECIAL,
    'forest': LandUse.RECREATION, 'grass': LandUse.RECREATION, 'recreation_ground': LandUse.RECREATION,
    'farmland': LandUse.AGRICULTURE, 'meadow': LandUse.AGRICULTURE, 'orchard': LandUse.AGRICULTURE,
}
urban_key = key(source_key, RULES, {k: CFG[k] for k in
                ('living_m2_person', 'default_floors', 'living_share', 'clusters', 'seed')})

def disjoint_by_priority(frame):
    # Индекс неизменяемых исходных контуров позволяет вычитать только пересекающихся предшественников.
    frame = frame.reset_index(drop=True).copy()
    if frame.empty:
        return frame
    previous = frame.sindex.query(frame.geometry, predicate='intersects')
    left, right = previous[:, previous[1] < previous[0]]
    neighbors = pd.Series(right, index=left).groupby(level=0).agg(list)
    original = frame.geometry.copy()
    for i, indexes in neighbors.items():
        frame.at[i, 'geometry'] = original.iloc[i].difference(original.iloc[indexes].union_all())
    return shapes(frame, POLY)

def build_urban():
    barriers = osm['barriers']
    surface = ~tag(barriers, 'tunnel').isin(['yes', 'building_passage']) & ~tag(barriers, 'bridge').isin(['yes', 'viaduct'])
    roads = shapes(barriers[surface & tag(barriers, 'highway').ne('')], LINE)
    rail = shapes(barriers[surface & tag(barriers, 'railway').ne('')], LINE)
    water = shapes(barriers[tag(barriers, 'natural').eq('water') | tag(barriers, 'waterway').ne('')], POLY + LINE)
    cutters, exclusions = preprocess_urban_objects(roads, rail, water)
    blocks = cut_urban_blocks(city, cutters, exclusions)
    require(not blocks.empty and blocks.geometry.is_valid.all() and (blocks.area > 0).all(),
            'Нет корректных кварталов')
    fabric = blocks.geometry.union_all()
    require(abs(blocks.area.sum() - fabric.area) < 1, 'Кварталы перекрываются')
    require(fabric.difference(city.geometry.iloc[0]).area < 1, 'Кварталы выходят за границу города')
    hashes = blocks.geometry.normalize().to_wkb().map(lambda b: sha256(b).hexdigest()[:24])
    require(hashes.is_unique, 'Найдены дубли кварталов')
    blocks['stable_id'] = hashes
    blocks = blocks.sort_values('stable_id').reset_index(drop=True)
    blocks.index.name = 'block_id'

    landscape = osm['landscape']
    landscape = landscape.assign(functional_zone=tag(landscape, 'landuse'))
    landscape.loc[tag(landscape, 'leisure').isin(['park', 'garden', 'recreation_ground']), 'functional_zone'] = 'recreation_ground'
    zones = shapes(landscape[landscape.functional_zone.isin(RULES)], POLY)
    zones = disjoint_by_priority(zones.assign(area_order=zones.area).sort_values(['area_order', 'osm_id']))
    if not zones.empty:
        land_use = assign_land_use(blocks[['geometry']], zones, RULES)
        blocks = blocks.join(land_use.drop(columns='geometry'))
        blocks['land_use'] = blocks['land_use'].map(lambda v: v.value if isinstance(v, LandUse) else v)
    else:
        blocks['land_use'] = None
        for category in LandUse:
            blocks[category.value] = 0.0
    known = blocks[[category.value for category in LandUse]].sum(axis=1)
    require(known.between(-1e-7, 1 + 1e-7).all(), 'Некорректные функциональные доли')
    blocks['land_use_unknown'] = (1 - known).clip(0, 1)

    buildings = shapes(osm['structures'], POLY)
    require(not buildings.empty, 'Нет контуров зданий в границе')
    buildings = buildings.sort_values('osm_id')
    buildings = buildings.loc[~buildings.geometry.normalize().to_wkb().duplicated()].copy()
    residential_zones = zones[zones.functional_zone.eq('residential')]
    points = buildings.set_geometry(buildings.representative_point())
    in_residential = points.sjoin(residential_zones[['geometry']], predicate='within').index.unique()
    building_type = tag(buildings, 'building')
    residential = building_type.isin(['apartments', 'house', 'detached', 'semidetached_house',
                                      'terrace', 'residential', 'dormitory', 'bungalow'])
    generic = building_type.isin(['yes', ''])
    floors = pd.to_numeric(tag(buildings, 'building:levels'), errors='coerce').astype(float)
    floors = floors.where(np.isfinite(floors) & floors.gt(0))
    buildings['floors_imputed'] = floors.isna().astype(int)
    buildings['use_unspecified'] = generic.astype(int)
    buildings['is_living'] = residential | (generic & buildings.index.isin(in_residential))
    buildings['number_of_floors'] = floors
    imputed = impute_buildings(buildings, default_number_of_floors=CFG['default_floors'],
                              default_living_demand=CFG['living_m2_person'], living_area_coefficient=CFG['living_share'])
    buildings = buildings.join(imputed.drop(columns=['geometry', 'is_living', 'number_of_floors']))
    buildings['number_of_floors'] = imputed.number_of_floors
    buildings['population'] = buildings.population.fillna(0).astype(float)
    measures = ['population', 'footprint_area', 'build_floor_area', 'living_area', 'non_living_area',
                'floors_imputed', 'use_unspecified']
    base = blocks[['geometry']].rename_axis(None).copy()
    aggregate, outside = aggregate_objects(base, buildings[measures + ['geometry']])
    require(np.allclose(aggregate[measures].sum() + outside[measures].sum(), buildings[measures].sum()),
            'Нарушен баланс при агрегации зданий')
    blocks = blocks.join(aggregate[measures + ['count']])
    blocks['site_area'] = blocks.area
    blocks['population_ha'] = safe_ratio(blocks.population, blocks.site_area / 10_000)
    density = calculate_density_indicators(blocks[['site_area', *measures[1:5]]].astype(float))
    metrics = ['fsi', 'gsi', 'mxi', 'l', 'osr', 'share_living', 'share_non_living']
    blocks = blocks.join(density[metrics].replace([np.inf, -np.inf], np.nan))
    developed = density.loc[density.fsi.gt(0), ['l', 'fsi', 'mxi']]
    require(len(developed) >= 2 and np.isfinite(developed).all().all(), 'Недостаточно данных о застройке')
    space, cluster = get_spacematrix_morphotypes(developed, n_clusters=min(CFG['clusters'], len(developed.drop_duplicates())),
                                                random_state=CFG['seed'])
    strelka = get_strelka_morphotypes(developed)
    blocks['spacematrix_cluster'] = space['cluster']
    blocks['spacematrix'] = space['morphotype'].astype('string')
    blocks['strelka'] = strelka['morphotype'].astype('string')
    require(blocks[['gsi', 'mxi']].stack().between(0, 1).all(), 'Недопустимая доля застройки')
    blocks['footprint_exceeds_site'] = blocks.footprint_area.gt(blocks.site_area)
    blocks.index.name = 'block_id'
    return dict(blocks=blocks, buildings=buildings, buildings_outside=outside, clusters=cluster,
                water=water, roads=roads, rail=rail)

urban = stage('urban', urban_key, build_urban)
blocks = urban['blocks'].copy()
blocks['population_original'] = blocks.population
blocks['population_ha_original'] = blocks.population_ha
population_factor = CFG['population_target'] / blocks.population_original.sum()
blocks['population'] = calibrate_population(blocks.population_original, CFG['population_target'])
blocks['population_ha'] = safe_ratio(blocks.population, blocks.site_area / 10_000)
display(blocks.drop(columns='geometry').describe(include='all').T)
for field, label in [('land_use', 'Функциональное использование'), ('population_ha', 'Жители на гектар'),
                     ('gsi', 'Доля пятен зданий'), ('fsi', 'Интенсивность застройки'), ('l', 'Этажность'),
                     ('spacematrix', 'Spacematrix'), ('strelka', 'Strelka')]:
    show_map(thematic(blocks, field, label), field)

## 3. Рекреационные ресурсы и зарегистрированные входы

Источники — OSM и, при наличии, проверенные локальные слои. В input/recreation.geojson можно передать проверенные полигоны общего пользования с полем name. При его наличии этот слой заменяет выборку полигонов OSM. В input/entrances.geojson можно добавить проверенные доступные входы. Линии набережных показываются как маркеры, площадь по ним не оценивается. В OSM отбираются парки, сады, рекреационные территории и явно обозначенные скверы, бульвары и набережные, исключаются ограниченные и специализированные территории. Отбор по тегам остаётся предварительным инвентарём.

Перекрытия вычитаются по убыванию площади. Входы привязываются к ближайшему контуру в пределах заданного допуска. Если входов нет, площадь видна в инвентаре, а доступность объекта остаётся непроверенной. Подготовьте проверенные входы, если охват OSM недостаточен.

После построения пешего графа выявляются потенциальные входы в местах пересечения пеших линий с границей ресурса. Для совпадающего по границе отрезка кандидатами служат его концы. Кандидаты не дублируют зарегистрированные или уже принятые точки в пределах 15 м по тому же объекту. Потенциальная точка не подтверждает возможность входа на местности. Тип сохраняется как observed или potential.

In [ ]:
recreation_file = INPUT / 'recreation.geojson'
entrances_file = INPUT / 'entrances.geojson'
def optional_hash(path):
    return file_hash(path) if path.exists() else None

recreation_key = key(urban_key, optional_hash(recreation_file), optional_hash(entrances_file),
                     CFG['entrance_tolerance_m'])

def public_access(frame):
    return ~tag(frame, 'access').isin(['private', 'no', 'customers', 'permit', 'members', 'residents'])

def tagged_recreation(frame):
    names = tag(frame, 'name').str.casefold()
    public_green = tag(frame, 'leisure').isin(['park', 'garden', 'recreation_ground'])
    named_space = names.str.contains(r'сквер|бульвар|набережн') & (
        tag(frame, 'place').eq('square') | tag(frame, 'footway').eq('promenade')
        | tag(frame, 'man_made').eq('embankment') | tag(frame, 'landuse').isin(['grass', 'recreation_ground', 'village_green']))
    specialized = tag(frame, 'garden:type').isin(['residential', 'community', 'allotment', 'private'])
    return (public_green | named_space) & public_access(frame) & ~specialized

def build_recreation():
    landscape = osm['landscape']
    if recreation_file.exists():
        resources = shapes(gpd.read_file(recreation_file), POLY)
        resources = resources[['geometry']].assign(name=tag(resources, 'name'), source='Проверенный локальный слой')
    else:
        resources = shapes(landscape[tagged_recreation(landscape)], POLY)
        resources = resources[['geometry']].assign(name=tag(resources, 'name'), source='OSM', osm_id=resources.osm_id)
    require(not resources.empty, 'Нет выявленных рекреационных полигонов')
    resources['input_area'] = resources.area
    resources['geometry_key'] = resources.geometry.normalize().to_wkb(hex=True)
    resources = disjoint_by_priority(resources.sort_values(['input_area', 'geometry_key'], ascending=[False, True]))
    resources = resources.drop(columns='geometry_key')
    resources['resource_id'] = np.arange(len(resources))
    resources['resource_area_m2'] = resources.area
    require(np.isclose(resources.area.sum(), resources.geometry.union_all().area, rtol=1e-9),
            'Рекреационные площади перекрываются')
    points = osm['access_points']
    points = shapes(points[public_access(points) & ~tag(points, 'entrance').isin(
        ['no', 'service', 'emergency', 'exit', 'private', 'home', 'garage'])], ['Point'])
    # Входы в здания не следует автоматически принимать за входы в парк.
    building_edges = urban['buildings'].geometry.boundary
    on_building = building_edges.sindex.query(points.geometry, predicate='dwithin', distance=2)[0]
    points = points.iloc[np.setdiff1d(np.arange(len(points)), on_building)][['geometry']].assign(source='OSM')
    if entrances_file.exists():
        extra = shapes(gpd.read_file(entrances_file), ['Point'])[['geometry']].assign(source='Проверенный локальный слой')
        points = pd.concat([extra, points], ignore_index=True)
    points = points.loc[~points.geometry.to_wkb().duplicated()].copy()
    boundaries = resources[['resource_id', 'geometry']].set_geometry(resources.boundary)
    entries = points.sjoin_nearest(boundaries, max_distance=CFG['entrance_tolerance_m'], distance_col='offset_m')
    entries = entries.sort_values(['offset_m', 'resource_id'])
    entries = entries.loc[~entries.index.duplicated()].drop(columns='index_right').reset_index(drop=True)
    resources['entry_count'] = resources.resource_id.map(entries.resource_id.value_counts()).fillna(0).astype(int)
    return dict(resources=resources, entrances=entries)

recreation = stage('recreation', recreation_key, build_recreation)
resources, entrances = recreation['resources'], recreation['entrances']
display(resources.groupby('source').agg(area_m2=('resource_area_m2', 'sum'), entrances=('entry_count', 'sum')))
show_map(map_layers([('Рекреационные территории', resources, '#2b9348'), ('Входы', entrances, '#1d3557')]), 'recreation_inventory')

## 4. Водно-зелёный каркас, маркеры и барьеры

Зелёная доля — площадь объединённых контуров растительности и выявленных рекреационных полигонов в квартале / площадь квартала. Это доля по картографическим контурам OSM, включая древесную растительность и открытые зелёные территории, а не измерение растительного покрытия. Историко-культурные, природные и рекреационные маркеры выбираются по тегам OSM. Концентрация измеряется количеством маркеров на км² квартала. Источник локальной границы поймы следует указать в конфигурации. Иначе используется условный буфер вокруг осей Урала и Сакмары. Доля пересечения определяет группу квартала.

Барьеры показаны по воде, железным дорогам и крупным дорогам. Мосты и переходы помогают интерпретировать проницаемость. Сравнение двух групп описательное, оно зависит от границы и полноты данных.

In [ ]:
flood_file = INPUT / 'floodplain.geojson'
identity_key = key(urban_key, recreation_key, optional_hash(flood_file), CFG['floodplain_source'],
                   CFG['riparian_buffer_m'], CFG['riparian_share_min'])

def area_in_blocks(frame):
    if frame.empty:
        return pd.Series(0.0, index=blocks.index)
    overlap = gpd.overlay(blocks[['geometry']].reset_index(), frame[['geometry']], keep_geom_type=True)
    return overlap.area.groupby(overlap.block_id).sum().reindex(blocks.index, fill_value=0)

def build_identity():
    land = osm['landscape']
    green = shapes(land[tag(land, 'natural').isin(['wood', 'grassland', 'scrub', 'wetland'])
                        | tag(land, 'landuse').isin(['forest', 'grass', 'meadow', 'village_green'])], POLY)
    green = polygon_frame(shp.get_parts(pd.concat([green.geometry, resources.geometry]).union_all()))
    streams = osm['barriers']
    river_axes = snapshot['river_axes']
    names = (tag(river_axes, 'name') + ' ' + tag(river_axes, 'name:ru')).str.casefold()
    axes = shapes(river_axes[names.str.contains(r'\bурал\b|\bсакмара\b')], LINE,
                  clip=False)
    if flood_file.exists():
        require(bool(CFG['floodplain_source']), 'Укажите источник границы поймы')
        flood = shapes(gpd.read_file(flood_file), POLY)
        label = 'Пойма по локальному источнику'
    else:
        require(all((tag(axes, 'name') + ' ' + tag(axes, 'name:ru')).str.casefold().str.contains(river).any()
                    for river in ('урал', 'сакмара')), 'В OSM не найдены оси обеих рек, предоставьте границу поймы')
        flood = polygon_frame([axes.geometry.union_all().buffer(CFG['riparian_buffer_m'])])
        label = 'Условный приречный буфер'
    require(not flood.empty, 'Граница поймы не пересекает город')
    flood = shapes(polygon_frame([flood.geometry.union_all()]), POLY).assign(zone_type=label)
    shares = pd.DataFrame(index=blocks.index)
    shares['riparian_share'] = safe_ratio(area_in_blocks(flood), blocks.site_area)
    shares['riparian'] = shares.riparian_share.ge(CFG['riparian_share_min'])
    shares['green_share'] = safe_ratio(area_in_blocks(green), blocks.site_area)
    shares['local_recreation_m2'] = area_in_blocks(resources)
    require(shares[['green_share', 'riparian_share']].stack().between(0, 1 + 1e-8).all(),
            'Проверьте пересечения кварталов')
    markers = osm['markers']
    cultural = tag(markers, 'historic').ne('') | tag(markers, 'heritage').ne('') | tag(markers, 'tourism').isin(['museum', 'artwork'])
    natural = tag(land, 'natural').isin(['peak', 'cliff', 'rock', 'spring', 'beach', 'tree']) & tag(land, 'name').ne('')
    candidates = [(markers[cultural], 'Историко-культурный'), (land[natural], 'Природный'),
                  (markers[~cultural & tag(markers, 'tourism').isin(['viewpoint', 'attraction'])], 'Видовой или туристический'),
                  (land[tagged_recreation(land)], 'Рекреационный')]
    parts = []
    if recreation_file.exists():
        pts = resources[['geometry', 'name']].set_geometry(resources.representative_point()).copy()
        pts['osm_id'] = 'local/' + resources.resource_id.astype(str)
        pts['category'] = 'Рекреационный'
        parts.append(pts)
    for frame, category in candidates:
        clipped = shapes(frame, POLY + LINE + ['Point'])
        pts = clipped[['osm_id', 'geometry']].set_geometry(clipped.representative_point())
        pts['name'] = tag(clipped, 'name')
        pts['category'] = category
        parts.append(pts)
    attrs = pd.concat(parts, ignore_index=True).drop_duplicates('osm_id')
    attrs = attrs.loc[~pd.DataFrame({'category': attrs.category, 'geometry': attrs.geometry.to_wkb()}).duplicated()]
    matched = attrs.sjoin(blocks[['geometry']].reset_index(), how='left', predicate='intersects').sort_values('block_id')
    attrs = matched.loc[~matched.index.duplicated()].drop(columns='index_right')
    shares['attractor_count'] = attrs.block_id.value_counts().reindex(blocks.index, fill_value=0)
    shares['attractors_km2'] = safe_ratio(shares.attractor_count, blocks.site_area / 1e6)

    road = urban['roads']
    blockers = ['motorway', 'trunk', 'primary', 'secondary']
    crossing_source = pd.concat([streams, osm['crossings']], ignore_index=True).drop_duplicates('osm_id')
    bridges = shapes(crossing_source[tag(crossing_source, 'bridge').isin(['yes', 'viaduct'])], LINE)
    crossings = shapes(crossing_source[tag(crossing_source, 'highway').eq('crossing')
                                        | tag(crossing_source, 'railway').isin(['crossing', 'level_crossing'])], ['Point'])
    barriers = pd.concat([road[tag(road, 'highway').isin(blockers)][['geometry']].assign(kind='Магистраль'),
                          urban['rail'][['geometry']].assign(kind='Железная дорога'),
                          urban['water'][['geometry']].assign(kind='Водный объект')], ignore_index=True)
    return dict(identity=shares, green=green, flood=flood, attractors=attrs, barriers=barriers, bridges=bridges, crossings=crossings)

identity = stage('identity', identity_key, build_identity)
blocks = blocks.join(identity['identity'])
show_map(map_layers([('Зелёные территории', identity['green'], '#287d38'), ('Вода', urban['water'], '#0088cc'),
                    ('Приречная зона', identity['flood'], '#7451a3'), ('Барьеры', identity['barriers'], '#862d2d'),
                    ('Мосты', identity['bridges'], '#704603'), ('Переходы', identity['crossings'], '#bd7013'), ('Аттракторы', identity['attractors'], '#d48815')]), 'water_green_framework')
show_map(thematic(blocks, 'attractors_km2', 'Аттракторов на км²'), 'attractors_km2')
show_map(thematic(blocks, 'green_share', 'Доля зелёных территорий'), 'green_share')

## 5. Транспортная матрица по локальному графу

По умолчанию используются автомобильный и пешеходный графы из PBF. Готовые transport.graphml и walk.graphml в input имеют приоритет. Для готового транспортного графа укажите источник, режим и веса. В автомобильной модели время рассчитано по длине сегмента и maxspeed или резервной скорости класса дороги. Модель учитывает базовые односторонние ограничения. Поворотные запреты, расписания и заторы не моделируются. Связность по совпадающим вершинам линий следует проверять у мостов, тоннелей и развязок. Поддерживаются минуты и секунды. Маршрутизация использует направленный граф и минимальный вес параллельных рёбер. Для режима транспорта используется время на графе между ближайшими узлами. Время подхода или ожидания учитывается только в той мере, в какой оно включено в исходные веса. Привязки дальше заданного допуска отмечаются как неизвестные. Матрица использует внутренние точки кварталов, как в BlocksNet. Маршрутизация через SciPy сохраняет все компоненты исходного графа и контролирует привязку. BlocksNet проверяет входной граф и рассчитывает взвешенное по площади время по заполненным строкам матрицы. Дополнительно выводится среднее по достижимой площади и её доля.

Диагностика разделяет связность сети и охват привязки кварталов. Дополнительный показатель reachable_bound_area_share использует в знаменателе только площадь кварталов, привязанных в пределах действующего порога. Исходная reachable_area_share по-прежнему делится на всю площадь кварталов.

In [ ]:
ready_transport = INPUT / 'transport.graphml'
walk_file = INPUT / 'walk.graphml'
graph_folder = CACHE / f'graphs-{source_key}'
ways = snapshot['network_ways']
transport_mode = CFG['transport_mode']
transport_source = CFG['transport_source']
transport_weight = CFG['transport_weight']
transport_unit = CFG['transport_unit']
if ready_transport.is_file():
    require(bool(transport_mode) and bool(transport_source),
            'Укажите transport_mode и transport_source для готового графа')
    transport_file = ready_transport
else:
    transport_file = ensure_graph(graph_folder / 'drive.graphml', ways, METRIC, 'drive')
    transport_mode = 'Автомобиль, расчётное время'
    transport_source = 'Локальный PBF BBBike'
    transport_weight, transport_unit = 'time_min', 'minutes'
walk_path = walk_file if walk_file.is_file() else ensure_graph(graph_folder / 'walk.graphml', ways, METRIC, 'walk')
walk_key = key(source_key, file_hash(walk_path), 'walk')
print(f'Графы: {transport_file} и {walk_path}')
entry_key = key(recreation_key, walk_key, quality_key)

def add_potential_entries():
    separated = (tag(ways, 'bridge').ne('') & tag(ways, 'bridge').ne('no')
                 | tag(ways, 'tunnel').ne('') & tag(ways, 'tunnel').ne('no')
                 | ~tag(ways, 'layer').isin(['', '0']))
    return infer_entrances(recreation['resources'], recreation['entrances'], walk_path,
                           set(ways.loc[separated, 'osm_id']), CFG['potential_entry_spacing_m'])

entry_inventory = stage('recreation-entries', entry_key, add_potential_entries)
resources, entrances = entry_inventory['resources'], entry_inventory['entrances']
show_map(map_layers([('Рекреационные территории', resources, '#2b9348'),
                     ('Зарегистрированные входы', entrances[entrances.entrance_kind.eq('observed')], '#1d3557'),
                     ('Потенциальные входы', entrances[entrances.entrance_kind.eq('potential')], '#e08214')]),
         'recreation_inventory')

require(transport_unit in ('minutes', 'seconds'), 'Время графа должно быть в минутах или секундах')
require(transport_mode.casefold() not in {'walk', 'walking', 'пешком', 'пешеходный'},
        'Транспортный и пешеходный графы должны соответствовать разным режимам')

class Routing:
    def __init__(self, graph, weight, factor=1):
        require(graph.number_of_nodes() > 0 and graph.number_of_edges() > 0, 'Граф пуст')
        require('crs' in graph.graph, 'В графе отсутствует CRS')
        graph_crs = CRS.from_user_input(graph.graph['crs'])
        self.nodes = {node: i for i, node in enumerate(graph.nodes)}
        try:
            xy = np.array([(float(data['x']), float(data['y'])) for _, data in graph.nodes(data=True)])
        except (KeyError, TypeError, ValueError) as error:
            raise ValueError('В узлах графа нужны числовые x и y') from error
        require(np.isfinite(xy).all(), 'В графе неконечные координаты')
        locations = gpd.GeoSeries(gpd.points_from_xy(*xy.T), crs=graph_crs).to_crs(METRIC)
        require(np.isfinite(locations.x).all() and np.isfinite(locations.y).all(), 'Проверьте CRS графа')
        self.tree = cKDTree(np.column_stack([locations.x, locations.y]))
        edges = {}
        for u, v, data in graph.edges(data=True):
            require(weight in data, f'На ребре {u} → {v} отсутствует {weight}')
            cost = float(data[weight]) * factor
            require(np.isfinite(cost) and cost >= 0, 'На графе некорректный вес')
            i, j = self.nodes[u], self.nodes[v]
            edges[i, j] = min(cost, edges.get((i, j), np.inf))
            if not graph.is_directed():
                edges[j, i] = min(cost, edges.get((j, i), np.inf))
        rows, cols = zip(*edges)
        self.csr = csr_matrix((list(edges.values()), (rows, cols)), shape=(len(xy), len(xy)))

    def snap(self, points):
        coordinates = np.column_stack([points.x, points.y])
        dist, pos = self.tree.query(coordinates)
        return pos, dist

def load_transport():
    graph = nx.read_graphml(transport_file)
    routing = Routing(graph, transport_weight, 1 / 60 if transport_unit == 'seconds' else 1)
    epsg = CRS.from_user_input(graph.graph['crs']).to_epsg()
    require(epsg is not None, 'Для готового графа нужен CRS с известным EPSG-кодом')
    graph.graph['crs'] = epsg
    validate_accessibility_graph(graph, transport_weight)
    return routing

transport_key = key(urban_key, file_hash(transport_file), transport_weight, transport_unit,
                    transport_mode, transport_source, CFG['snap_limit_m'], CFG['max_matrix_mib'])
origins = blocks.geometry.representative_point()

def build_transport():
    budget = CFG['max_matrix_mib'] * 1024**2
    n = len(blocks)
    require(12 * n * n < budget, 'Матрица кварталов превышает лимит памяти; пересмотрите max_matrix_mib')
    route = load_transport()
    require(route.csr.shape[0] * 16 + 12 * n * n < budget, 'Недостаточно памяти для маршрутизации')
    positions, offsets = route.snap(origins)
    good = offsets <= CFG['snap_limit_m']
    require(good.any(), 'Ни один квартал не привязан к транспортному графу')
    require(good.mean() >= 0.5, 'Транспортный граф покрывает менее половины кварталов. Проверьте охват и CRS')
    times = np.full((n, n), np.inf, dtype=np.float32)
    selected = np.flatnonzero(good)
    # Один поиск на уникальный узел отправления. Память поиска ограничена одной строкой.
    groups = pd.Series(selected, index=positions[selected]).groupby(level=0).agg(list)
    for node, rows in groups.items():
        distances = dijkstra(route.csr, indices=int(node))[positions[selected]]
        times[np.ix_(rows, selected)] = distances
    require(np.allclose(np.diag(times)[good], 0), 'Ненулевая диагональ привязанных кварталов')
    weights = blocks.site_area.to_numpy()
    metrics = pd.DataFrame({'transport_snap_m': offsets, 'transport_bound': good}, index=blocks.index)
    complete = []
    reachable_share, accessible = [], []
    for row in times:
        finite = np.isfinite(row)
        covered = weights[finite].sum()
        reachable_share.append(covered / weights.sum())
        accessible.append(np.dot(row[finite], weights[finite]) / covered if covered else np.nan)
        complete.append(finite.all())
    metrics['reachable_area_share'] = reachable_share
    metrics['reachable_area_time_min'] = accessible
    matrix = pd.DataFrame(times, index=blocks.index, columns=blocks.index.astype(str))
    metrics['area_accessibility'] = np.nan
    if any(complete):
        rows = matrix.loc[complete].rename(columns=int)
        metrics.loc[rows.index, 'area_accessibility'] = area_accessibility(rows, blocks, out=True).iloc[:, 0]
    return dict(matrix=matrix, transport_metrics=metrics)

transport = stage('transport', transport_key, build_transport)
matrix = transport['matrix']
require(matrix.index.equals(blocks.index) and list(matrix.columns) == list(blocks.index.astype(str)),
        'Матрица не соответствует кварталам')
blocks = blocks.join(transport['transport_metrics'])
bound_area_share = blocks.loc[blocks.transport_bound, 'site_area'].sum() / blocks.site_area.sum()
blocks['reachable_bound_area_share'] = (blocks.reachable_area_share / bound_area_share).where(blocks.transport_bound)
require(blocks.reachable_bound_area_share.dropna().between(0, 1 + 1e-8).all(),
        'Доля достижимой привязанной площади вне допустимого диапазона')

def diagnose_transport():
    routing = load_transport()
    summary = {}
    for connection in ('weak', 'strong'):
        count, labels = connected_components(routing.csr, directed=True, connection=connection)
        sizes = np.bincount(labels)
        summary[f'{connection}_components'] = count
        summary[f'largest_{connection}_component_nodes'] = int(sizes.max())
    summary['graph_nodes'] = routing.csr.shape[0]
    summary['snap_limit_m'] = CFG['snap_limit_m']
    summary['blocks_total'] = len(blocks)
    summary['blocks_bound'] = int(blocks.transport_bound.sum())
    summary['bound_area_share'] = bound_area_share
    summary['median_reachable_area_share'] = blocks.reachable_area_share.median()
    summary['median_reachable_bound_area_share'] = blocks.reachable_bound_area_share.median()
    summary['largest_unbound_block_m2'] = blocks.loc[~blocks.transport_bound, 'site_area'].max()
    return {'diagnostics': pd.Series(summary, name='value').to_frame()}

diagnostics_key = key(transport_key, 'connectivity-diagnostics-v1')
transport_diagnostics = stage('transport-diagnostics', diagnostics_key, diagnose_transport)['diagnostics']
display(transport_diagnostics)
display(blocks[['transport_bound', 'reachable_area_share', 'reachable_area_time_min']].describe(include='all'))
show_map(thematic(blocks, 'reachable_area_time_min', 'Среднее до достижимой площади, мин'), 'reachable_area_time_min')
show_map(thematic(blocks, 'reachable_area_share', 'Доля достижимой площади'), 'reachable_area_share')

## 6. Рекреационная обеспеченность

Рабочие параметры исследования по таблице 36 раздела 1.16 — 11 м²/чел, 1000 м пешком до входов и 10 минут на транспорте. Последняя найденная редакция от 03.11.2026 не меняет таблицу 36 напрямую. Проверка всех поправок 2025 года осталась неполной, поэтому параметры фиксируются как рабочие. Подробности и ссылки в плане.

Пешая доступность считается по длине маршрута, включая расстояния привязки. По умолчанию пешая сеть строится из локального PBF. Можно передать готовый walk.graphml с атрибутом length в метрах. Для транспорта используются времена готового графа. В каждом сценарии применяется минимальная стоимость до одного из картированных входов объекта, в пределах порога. Для ресурсов без входов доступность неизвестна. Дефицит — остаток спроса в модели при имеющихся данных. Проверьте качество входов и охват графа прежде, чем интерпретировать его.

Алгоритм BlocksNet получает явные спрос и вместимость в целых м². Спрос округляется вверх, ресурс вниз. У каждого ресурса есть отдельный узел распределения, поэтому несколько входов не увеличивают вместимость. Автоматическое самообеспечение отключено. Каждый сценарий перераспределяет одну и ту же площадь независимо. Сценарии анализируются по отдельности.

| Поле | Смысл |
|---|---|
| recreation_demand_m2 | Оценка населения × 11 м²/чел |
| resource_area_m2 | Уникальная площадь выявленного контура, м² |
| local_recreation_m2 | Выявленная площадь внутри квартала, м² |
| local_m2_person | Площадь внутри квартала на жителя, м²/чел |
| walk_nearest, transit_nearest | Наименьшая стоимость до входа в пределах порога, м или мин. Пропуск означает отсутствие подтверждённого маршрута в пределах порога |
| walk_assigned_m2, transit_assigned_m2 | Площадь, распределённая кварталу по модели, м² |
| *_m2_person | Распределённая площадь на жителя, м²/чел |
| *_satisfied | Распределённая площадь / точная потребность, с верхней границей 100% |
| *_deficit_m2 | Точная потребность минус выделенная площадь, минимум 0 |
| *_origin_bound | Есть допустимая привязка квартала к графу |

Основной расчёт использует зарегистрированные и потенциальные входы. В итоговой таблице маршрутов к объекту сохраняется тип входа с минимальной стоимостью. При равной стоимости преимущество у зарегистрированного входа.

In [ ]:
def load_walk():
    return Routing(nx.read_graphml(walk_path), 'length')

def route_to_resources(route, limit, walking):
    columns = ['block_id', 'resource_id', 'cost', 'entrance_id', 'entrance_kind']
    source_pos, source_dist = route.snap(origins)
    good = source_dist <= CFG['snap_limit_m']
    result_status = pd.Series(good, index=blocks.index)
    if entrances.empty:
        return pd.DataFrame(columns=columns), result_status, np.empty(0, dtype=bool)
    entry_pos, entry_dist = route.snap(entrances.geometry)
    entry_good = entry_dist <= CFG['snap_limit_m']
    entry_ids = entrances.resource_id.to_numpy()[entry_good]
    entry_index = entrances.index.to_numpy()[entry_good]
    entry_kinds = entrances.entrance_kind.to_numpy()[entry_good]
    entry_pos, entry_dist = entry_pos[entry_good], entry_dist[entry_good]
    selected = np.flatnonzero(good)
    pairs = []
    label_count = 0
    groups = pd.Series(selected, index=source_pos[selected]).groupby(level=0).agg(list)
    for node, rows in groups.items():
        base = dijkstra(route.csr, indices=int(node), limit=limit)[entry_pos]
        if walking:
            base = base + entry_dist
        for pos in rows:
            cost = base + (source_dist[pos] if walking else 0)
            mask = cost <= limit
            if mask.any():
                minimum = pd.DataFrame({'resource_id': entry_ids[mask], 'cost': cost[mask],
                                        'entrance_id': entry_index[mask], 'entrance_kind': entry_kinds[mask]})
                minimum = minimum.sort_values(['cost', 'entrance_kind', 'entrance_id']).drop_duplicates('resource_id')
                label_count += len(minimum)
                require(label_count <= CFG['max_route_labels'], 'Превышен лимит допустимых пар маршрутов')
                pairs.append(minimum.assign(block_id=blocks.index[pos])[columns])
    return (pd.concat(pairs, ignore_index=True) if pairs else pd.DataFrame(columns=columns)), result_status, entry_good

def allocate(pairs, limit, demand, capacity):
    columns = ['block_id', 'resource_id', 'allocated_m2', 'cost', 'entrance_id', 'entrance_kind']
    valid = pairs.loc[pairs.block_id.isin(demand[demand.gt(0)].index)
                      & pairs.resource_id.isin(capacity[capacity.gt(0)].index)].copy()
    if valid.empty:
        return pd.DataFrame(columns=columns)
    a = pd.Index(sorted(valid.block_id.unique()))
    b = pd.Index(sorted(valid.resource_id.unique()))
    na, nb = len(a), len(b)
    require(na * nb <= CFG['max_solver_pairs'], 'Лимит пар для оптимизатора BlocksNet превышен')
    require(16 * (na + nb)**2 <= CFG['max_matrix_mib'] * 1024**2,
            'Служебная матрица распределения превышает лимит памяти')
    # Целые ID нужны для разбора переменных PuLP внутри BlocksNet 1.0.0a9.
    idx = pd.RangeIndex(na + nb)
    costs = np.full((len(idx), len(idx)), np.inf, dtype=np.float64)
    row = a.get_indexer(valid.block_id)
    col = b.get_indexer(valid.resource_id) + na
    costs[row, col] = valid.cost.to_numpy()
    np.fill_diagonal(costs, 0)
    demand_capacity = pd.DataFrame({'demand': np.r_[demand.loc[a].to_numpy(), np.zeros(nb, dtype=int)],
                                    'capacity': np.r_[np.zeros(na, dtype=int), capacity.loc[b].to_numpy()]},
                                   index=idx).astype('int64')
    provision, links = competitive_provision(demand_capacity, pd.DataFrame(costs, index=idx, columns=idx),
                                            accessibility=limit, self_supply=False, max_depth=1)
    require(np.allclose(provision.demand_left + provision.demand_within, provision.demand)
            and np.allclose(provision.capacity_left + provision.capacity_within, provision.capacity),
            'Нарушен баланс площади')
    table = links.reset_index()
    if table.empty:
        return pd.DataFrame(columns=columns)
    table['block_id'] = a.take(table.source.to_numpy(dtype=int))
    table['resource_id'] = b.take(table.target.to_numpy(dtype=int) - na)
    table = table.rename(columns={'value': 'allocated_m2'}).merge(
        valid, on=['block_id', 'resource_id'], validate='one_to_one')
    require(table.cost.le(limit).all() and table.allocated_m2.gt(0).all(), 'Распределение вне порога')
    used = table.groupby('resource_id').allocated_m2.sum()
    supplied = table.groupby('block_id').allocated_m2.sum()
    require(used.le(capacity.loc[used.index]).all() and supplied.le(demand.loc[supplied.index]).all(),
            'Распределено больше ресурса или спроса')
    return table[columns]

provision_key = key(identity_key, transport_key, walk_key, entry_key, quality_key, {k: CFG[k] for k in (
    'walk_limit_m', 'transport_limit_min', 'recreation_m2_person', 'max_solver_pairs', 'max_route_labels')})

def build_provision():
    metrics = pd.DataFrame(index=blocks.index)
    metrics['recreation_demand_m2'] = blocks.population * CFG['recreation_m2_person']
    metrics['local_m2_person'] = safe_ratio(blocks.local_recreation_m2, blocks.population)
    demand = np.ceil(metrics.recreation_demand_m2).astype('int64')
    capacity = np.floor(resources.set_index('resource_id').resource_area_m2).astype('int64')
    result = {}
    for prefix, loader, limit in (('walk', load_walk, CFG['walk_limit_m']),
                                  ('transit', load_transport, CFG['transport_limit_min'])):
        pairs, bound, usable_entries = route_to_resources(loader(), limit, walking=prefix == 'walk')
        links = allocate(pairs, limit, demand, capacity)
        metrics[f'{prefix}_origin_bound'] = bound
        metrics[f'{prefix}_nearest'] = pd.to_numeric(pairs.groupby('block_id').cost.min()).reindex(blocks.index)
        inventory_observed = bound & usable_entries.any()
        result[f'{prefix}_entry_audit'] = pd.DataFrame({
            'entrance_id': entrances.index, 'resource_id': entrances.resource_id, 'entrance_kind': entrances.entrance_kind, 'graph_bound': usable_entries,
        })
        assigned = pd.to_numeric(links.groupby('block_id').allocated_m2.sum()).reindex(blocks.index, fill_value=0)
        metrics[f'{prefix}_assigned_m2'] = assigned.where(inventory_observed)
        metrics[f'{prefix}_m2_person'] = safe_ratio(assigned, blocks.population).where(inventory_observed)
        metrics[f'{prefix}_satisfied'] = safe_ratio(assigned, metrics.recreation_demand_m2).clip(0, 1).where(inventory_observed)
        metrics[f'{prefix}_deficit_m2'] = (metrics.recreation_demand_m2 - assigned).clip(lower=0).where(inventory_observed)
        result[f'{prefix}_routes'] = pairs
        result[f'{prefix}_allocation'] = links
    return dict(metrics=metrics, **result)

provision = stage('provision', provision_key, build_provision)
blocks = blocks.join(provision['metrics'])
summary = pd.Series({
    'Население (исходная оценка), чел': blocks.population_original.sum(),
    'Население (после калибровки), чел': blocks.population.sum(),
    'Коэффициент калибровки': population_factor,
    'Объекты с зарегистрированными входами': resources.observed_entry_count.gt(0).sum(),
    'Объекты с потенциальными входами': resources.potential_entry_count.gt(0).sum(),
    'Потребность, м²': blocks.recreation_demand_m2.sum(),
    'Выявленная площадь, м²': resources.resource_area_m2.sum(),
    'Площадь объектов с входами, м²': resources.loc[resources.entry_count.gt(0), 'resource_area_m2'].sum(),
    'Выявленная площадь на жителя, м²/чел': resources.resource_area_m2.sum() / blocks.population.sum()
    if blocks.population.sum() else np.nan,
})
display(summary)
for mode, label in [('walk', 'Пешком'), ('transit', transport_mode)]:
    audit = provision[f'{mode}_entry_audit']
    observed_ids = audit.loc[audit.graph_bound, 'resource_id']
    print(f'{label}: входов с привязкой — {audit.graph_bound.sum()} из {len(audit)}')
    print(f'Площадь объектов с привязанными входами, м²: {resources.loc[resources.resource_id.isin(observed_ids), "resource_area_m2"].sum():,.0f}')
    show_map(thematic(blocks, f'{mode}_nearest', f'{label}: доступность в пределах порога'), f'{mode}_nearest')
    show_map(thematic(blocks, f'{mode}_satisfied', f'{label}: доля удовлетворённой потребности'), f'{mode}_satisfied')
    show_map(thematic(blocks, f'{mode}_deficit_m2', f'{label}: дефицит, м²'), f'{mode}_deficit_m2')

## 7. Сопоставление приречных кварталов и экспорт для практики № 3

Сравните размеры групп, оценку населения, площадь и медианы квартальных показателей. Городские доли и показатели на жителя вычисляйте по суммам, с явным знаменателем. При сравнении доступности сопоставьте охват графов и долю картированных входов. Данные сохраняются в WGS 84 для карт и в исходной проекции в кэше. В матрице строки и столбцы соответствуют кварталам. Недостижимые пары содержат бесконечность, а в GeoJSON неопределённые значения записываются как null.

In [ ]:
groups = blocks.assign(group=np.where(blocks.riparian, 'Приречные', 'Остальные')).groupby('group')
comparison = groups.agg(
    blocks=('stable_id', 'size'), population=('population', 'sum'), population_original=('population_original', 'sum'), area_m2=('site_area', 'sum'),
    population_ha_median=('population_ha', 'median'), fsi_median=('fsi', 'median'),
    gsi_median=('gsi', 'median'), floors_median=('l', 'median'),
    attraction_density_median=('attractors_km2', 'median'), green_share_median=('green_share', 'median'),
    transport_time_median=('reachable_area_time_min', 'median'), transport_area_share_median=('reachable_area_share', 'median'),
    transport_bound_area_share_median=('reachable_bound_area_share', 'median'),
    walk_satisfied_median=('walk_satisfied', 'median'), transit_satisfied_median=('transit_satisfied', 'median'),
    walk_observed=('walk_satisfied', 'count'), transit_observed=('transit_satisfied', 'count'),
    local_recreation_m2=('local_recreation_m2', 'sum'), attractors=('attractor_count', 'sum'),
)
comparison['recreation_m2_person'] = safe_ratio(comparison.local_recreation_m2, comparison.population)
comparison['attractors_km2'] = safe_ratio(comparison.attractors, comparison.area_m2 / 1e6)
for mode in ('walk', 'transit'):
    comparison[f'{mode}_deficit_m2'] = groups[f'{mode}_deficit_m2'].sum(min_count=1)
    comparison[f'{mode}_population_bound'] = blocks.population.where(blocks[f'{mode}_origin_bound']).groupby(
        np.where(blocks.riparian, 'Приречные', 'Остальные')).sum(min_count=1)
for mode in ('walk', 'transit'):
    denominator = blocks.recreation_demand_m2.where(blocks[f'{mode}_origin_bound']).groupby(
        np.where(blocks.riparian, 'Приречные', 'Остальные')).sum(min_count=1)
    comparison[f'{mode}_satisfied_weighted'] = 1 - safe_ratio(comparison[f'{mode}_deficit_m2'], denominator)
if len(comparison) < 2:
    print('Граница и порог сформировали только одну группу; сравнение невозможно')
display(comparison.T)
display(comparison[['green_share_median', 'walk_satisfied_median', 'transit_satisfied_median']].style.bar(vmin=0, vmax=1))
categories = pd.crosstab(blocks.riparian, blocks.spacematrix.fillna('Нет морфотипа'))
display(categories.style.bar(axis=1))

def export_geo(frame, path):
    data = frame.to_crs(4326)
    temporary = path.with_suffix('.tmp')
    temporary.write_text(data.to_json(na='null', drop_id=True, ensure_ascii=False), encoding='utf-8')
    temporary.replace(path)

require(blocks.index.is_unique and blocks.stable_id.is_unique and blocks.geometry.is_valid.all(),
        'Идентификаторы или геометрия кварталов нарушены')
require(blocks.population.sum() > 0, 'Не получена оценка населения. Проверьте жилые теги и геометрию.')
require(resources.resource_id.is_unique and entrances.resource_id.isin(resources.resource_id).all(),
        'Невалидные идентификаторы ресурсов или входов')
output_layers = {
    'blocks': blocks.reset_index(), 'city': city, 'resources': resources, 'entrances': entrances.reset_index(),
    'attractors': identity['attractors'], 'green': identity['green'], 'riparian': identity['flood'],
    'water': urban['water'], 'barriers': identity['barriers'], 'bridges': identity['bridges'], 'crossings': identity['crossings'],
}
tables = {'comparison': comparison, 'spacematrix_clusters': urban['clusters'],
          'city_summary': summary.rename('value').to_frame(), 'transport_diagnostics': transport_diagnostics,
          'blocks_attributes': blocks.drop(columns='geometry'),
          **{name: frame for name, frame in provision.items() if name != 'metrics'}}
export_signature = key(provision_key, CFG)
output_folder = OUT / export_signature
output_folder.mkdir(exist_ok=True)
manifest = output_folder / 'metadata.json'
expected = {f'{name}.geojson' for name in output_layers} | {f'{name}.csv' for name in tables} | {'transport_matrix_min.parquet'}
try:
    prior = json.loads(manifest.read_text(encoding='utf-8'))
    cached = (prior['signature'] == export_signature and set(prior['files']) == expected
              and all(file_hash(output_folder / path) == checksum for path, checksum in prior['files'].items()))
except (OSError, ValueError, KeyError):
    cached = False
if not cached:
    for name, frame in output_layers.items():
        export_geo(frame, output_folder / f'{name}.geojson')
    for name, table in tables.items():
        table.to_csv(output_folder / f'{name}.csv', encoding='utf-8-sig',
                     index=name in {'comparison', 'spacematrix_clusters', 'city_summary', 'blocks_attributes', 'transport_diagnostics'})
    matrix.to_parquet(output_folder / 'transport_matrix_min.parquet')
    restored = gpd.read_file(output_folder / 'blocks.geojson').set_index('block_id').reindex(blocks.index)
    require(np.array_equal(restored.stable_id.to_numpy(), blocks.stable_id.to_numpy()) and restored.geometry.notna().all()
            and np.allclose(restored.site_area, blocks.site_area), 'Проверка GeoJSON после записи не пройдена')
    metadata = dict(
        signature=export_signature,
        files={name: file_hash(output_folder / name) for name in sorted(expected)},
        parameters=CFG, versions=VERSIONS, metric_crs=METRIC.to_string(),
        population_calibration={'original': float(blocks.population_original.sum()),
                                'target': CFG['population_target'], 'factor': population_factor,
                                'source': 'Статистическая численность, заданная пользователем'},
        access_points={'observed': int(entrances.entrance_kind.eq('observed').sum()),
                       'potential': int(entrances.entrance_kind.eq('potential').sum()),
                       'method': 'Пересечения пешей сети с границей объекта'},
        territory_source=str(city_file) if city_file.exists() else 'OSM Nominatim',
        osm_snapshot={'path': str(snapshot_file), 'sha256': snapshot_sha256, 'provider': 'BBBike'},
        transport_graph={'path': str(transport_file), 'source': transport_source, 'mode': transport_mode,
                         'weight': transport_weight, 'unit': transport_unit,
                         'fallback_speed_kmh': DRIVE_SPEEDS if transport_file != ready_transport else None},
        walk_graph=str(walk_path),
        osm_decoded_utc={name: sorted(frame.retrieved_utc.dropna().unique().tolist()) for name, frame in osm.items()},
        graph_sha256=file_hash(transport_file), walk_graph_sha256=file_hash(walk_path),
        input_sha256={p.name: optional_hash(p) for p in (city_file, recreation_file, entrances_file, flood_file)},
        norms={'checked_on': '2026-11-18', 'status': 'рабочие параметры, сверка редакций 2025 неполна'},
        riparian_zone=identity['flood'].zone_type.iloc[0],
        unmatched_buildings=len(urban['buildings_outside']),
        uncovered_recreation_area_m2=float(resources.loc[resources.entry_count.eq(0), 'resource_area_m2'].sum()),
        matrix={'rows': 'block_id', 'columns': 'block_id as string', 'unit': 'minutes', 'unreachable': 'inf'},
        stage_keys=dict(osm=source_key, urban=urban_key, recreation=recreation_key, identity=identity_key,
                        transport=transport_key, provision=provision_key),
    )
    temporary = manifest.with_suffix('.tmp')
    temporary.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')
    temporary.replace(manifest)
print(f'Готовые файлы: {output_folder}')

## 8. Как прочитать результаты

1. Проверьте границу города, границу поймы или условного буфера и здания, не попавшие в кварталы.
2. Сверьте отмеченные входы на нескольких крупных рекреационных объектах. Проверьте вручную пеший путь и транспортный маршрут к одному и тому же входу, например в кварталах на противоположных берегах Урала.
3. Оцените рекреационную площадь без входов и число жилых кварталов без надёжной привязки к сети. Изучайте дефицит вместе с этими величинами.
4. Сравните приречные кварталы с остальным городом по нескольким показателям из таблицы. Проверьте на карте концентрации, переходы и пространственные барьеры. Укажите, какие различия зависят от условной границы или полноты OSM.
5. Для практики № 3 используйте папку, указанную в последней ячейке. При повторном запуске с теми же входами читается проверенный кэш. После правок входных файлов зависимые этапы пересчитаются. При замене PBF его контрольная сумма создаёт новый ключ кэша. Сохраните старый PBF для воспроизводимости. Для анализа динамики сохраните старые исходники и метаданные.